<a href="https://colab.research.google.com/github/rsher60/LLM_Codebase/blob/main/finetuning_udemy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U transformers datasets seaborn bertviz umap-learn

In [ ]:
from datasets import load_dataset

dataset = load_dataset('dair-ai/emotion')

In [ ]:
dataset

In [ ]:
df = dataset['train'].to_pandas()

In [ ]:
df.head()

In [ ]:
label_names = dataset['train'].features

In [ ]:
label_names['label']

In [ ]:
a = zip([0,1,2,3,4,5], ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise'])

In [ ]:
df.dtypes

In [ ]:
dict(a)

In [ ]:
# Convert 'label' column to integer type
df['label'] = df['label'].astype(int)

# Now map
a = zip([0, 1, 2, 3, 4, 5], ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise'])
df['mapped_label'] = df['label'].map(dict(a))

In [ ]:
df.groupby(['mapped_label']).count()

## Tokenization of Data from the Raw String

In [ ]:
from transformers import AutoTokenizer

model_checkpoint= "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

text = "My name is Riddhiman Sherlekar"
encoded_text = tokenizer(text)
print(encoded_text)

## Split the Data as :

1) Training Data
2) Test Data
3) Validation Data

In [ ]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(df, test_size = 0.3, stratify=df['mapped_label'])
test, validation = train_test_split(test, test_size = 1/3, stratify = test['label'])


train.shape , test.shape , validation.shape, df.shape

In [ ]:
dataset

In [ ]:
from datasets import Dataset, DatasetDict

dataset = DatasetDict({
    'train' : Dataset.from_pandas(train, preserve_index = False),
    'test' : Dataset.from_pandas(test, preserve_index = False),
    'validation' : Dataset.from_pandas(validation, preserve_index = False)
})

dataset

In [ ]:
dataset['train'][0] , dataset['test'][0] , dataset['validation'][0]

Tokenize all the text in Dataset using the Map function of HuggingFace

In [ ]:
def tokenize(batch):
  return tokenizer(batch['text'], padding= True, truncation=True)

tokenize(dataset['train'][:3])

In [ ]:
encoded = dataset.map(tokenize, batched=True, batch_size = None)

In [ ]:
encoded

In [ ]:
label_names['label']

In [ ]:
label_names = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']

Working with models

In [ ]:
from transformers import AutoModel
model = AutoModel.from_pretrained(model_checkpoint)

model.config.id2label, model.config.label2id

label2id = {label: i for i , label in enumerate(label_names)}
id2label = {i: label for i, label in enumerate(label_names)}


label2id, id2label

In [ ]:
model

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, AutoConfig
import torch

if torch.has_mps:
  device = torch.device("mps")
elif torch.cuda.is_available():
  device = torch.device("cuda")
else:
  device = torch.device("cpu")


print(device)

config = AutoConfig.from_pretrained(model_checkpoint, label2id = label2id , id2label = id2label)
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, config=config).to(device)




In [ ]:
#!pip install accelerate

!pip show accelerate

In [ ]:
from transformers import TrainingArguments
batch_size = 64
training_dir = "bert_base_uncased_trained_model"


training_args = TrainingArguments(output_dir=training_dir, overwrite_output_dir=True,
                                  num_train_epochs=2,
                                  learning_rate= 2e-5,
                                  per_device_eval_batch_size = batch_size,
                                  per_device_train_batch_size = batch_size,
                                  weight_decay = 0.01,
                                  eval_strategy='epoch',
                                  disable_tqdm=False)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(pred):
  labels = pred.label_ids
  preds = pred.predictions.argmax(-1)

  f1 = f1_score(labels, preds , average="weighted")
  acc = accuracy_score(labels, preds)

  return {"accuracy" : acc, "F1 score": f1}

 Build the Compute Metrics

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model = model ,
    args = training_args,
    compute_metrics = compute_metrics,
    train_dataset = encoded['train'],
    eval_dataset = encoded['validation'],
    tokenizer = tokenizer
)

In [ ]:
trainer.train()

Save the model

In [ ]:
trainer.save_model("bert_base_uncased_text-classification-model")

In [ ]:
from transformers import pipeline

classifier = pipeline('text-classification' , model="bert_base_uncased_text-classification-model")


classifier([
    'I like ML',
    'I started annoyed with laptops',
    'I am low today',
    'I am tensed if I am not going to get the job in ML',
    'I am worried with the current job market'





])

In [ ]:
from transformers import pipeline, AutoModel, AutoTokenizer

# Load your pipeline
classifier = pipeline('text-classification', model="bert_base_uncased_text-classification-model")

# Extract the model and tokenizer
model = classifier.model
tokenizer = classifier.tokenizer


# Push model and tokenizer
model.push_to_hub("rsher60/bert_base_uncased_text-classification-model")
tokenizer.push_to_hub("rsher60/bert_base_uncased_text-classification-model")

## Inference the model from HF

In [ ]:
from transformers import pipeline

classifier =